# Handwritten Digit Recognition Using SVM — Course Project (scikit-learn version)
Uses the built-in `sklearn.svm.SVC`. Run the cells top to bottom.

## 0. Setup and upload files
Upload: `training_1.txt`, `test_1.txt`, `training_2.txt`, `test_2.txt`, `training_3.text`, `train-01-images.svm`, `test-01-images.svm`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, glob, time
from sklearn.svm import SVC
from sklearn.datasets import load_svmlight_file
from sklearn.model_selection import StratifiedKFold, cross_val_score
try:
    from google.colab import files
    files.upload()
except ImportError:
    pass  # running locally

def find(*pats):
    for p in pats:
        m = glob.glob(p) + glob.glob("/mnt/user-data/uploads/" + p)
        if m: return m[0]
    raise FileNotFoundError(pats)

f_tr1, f_te1 = find("training_1.*", "train_1.*"), find("test_1.*")
f_tr2, f_te2 = find("training_2.*", "train_2.*"), find("test_2.*")
f_tr3        = find("training_3.*", "train_3.*")
f_trimg, f_teimg = find("train-01-images.svm"), find("test-01-images.svm")

def load_xy(fn):
    d = np.loadtxt(fn)
    return d[:, :2], np.where(d[:, 2] > 0, 1, -1)

def err(model, X, y): return float(np.mean(model.predict(X) != y))

def plot_boundary(clf, X, y, title, ax=None, margin=True, show_sv=False):
    ax = ax or plt.gca()
    pad = 0.08 * (X.max(0) - X.min(0))
    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-pad[0], X[:,0].max()+pad[0], 300),
                         np.linspace(X[:,1].min()-pad[1], X[:,1].max()+pad[1], 300))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z > 0, alpha=0.15, cmap="coolwarm")
    ax.contour(xx, yy, Z, levels=[0], colors="k", linewidths=2)
    if margin: ax.contour(xx, yy, Z, levels=[-1, 1], colors="k", linestyles="--", linewidths=1)
    ax.scatter(X[y==1,0],  X[y==1,1],  c="tab:red",  s=12, label="+1")
    ax.scatter(X[y==-1,0], X[y==-1,1], c="tab:blue", s=12, label="-1")
    if show_sv: ax.scatter(*clf.support_vectors_.T, s=60, facecolors="none", edgecolors="k", label="SV")
    ax.set_title(title, fontsize=10); ax.legend()

# Part 1 — Soft-margin SVM on the two 2-D datasets

In [ ]:
Cs1 = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100, 1000]
for k, (ftr, fte) in enumerate([(f_tr1, f_te1), (f_tr2, f_te2)], start=1):
    Xtr, ytr = load_xy(ftr); Xte, yte = load_xy(fte)
    print(f"================ Dataset {k}: {len(ytr)} train / {len(yte)} test ================")
    clf = SVC(kernel="linear", C=1).fit(Xtr, ytr)

    # (i) decision boundary on training data, (ii) test error
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    plot_boundary(clf, Xtr, ytr, f"Dataset {k} – training data, C=1", ax=axs[0], show_sv=True)
    plot_boundary(clf, Xte, yte, f"Dataset {k} – test data, C=1", ax=axs[1])
    plt.show()
    print("w =", clf.coef_[0], " b =", clf.intercept_[0], " margin = %.2f" % (2/np.linalg.norm(clf.coef_)))
    print("training error:", err(clf, Xtr, ytr))
    print("TEST error (fraction misclassified):", err(clf, Xte, yte))

    # (iii) different C
    rows = []
    fig, axs = plt.subplots(2, 5, figsize=(22, 8))
    for c, ax in zip(Cs1, axs.ravel()):
        m = SVC(kernel="linear", C=c).fit(Xtr, ytr)
        rows.append((c, err(m, Xtr, ytr), err(m, Xte, yte), len(m.support_), 2/np.linalg.norm(m.coef_)))
        plot_boundary(m, Xtr, ytr, f"C={c:g}  (#SV={len(m.support_)})", ax=ax)
    plt.suptitle(f"Dataset {k}: effect of C"); plt.tight_layout(); plt.show()
    print(f"{'C':<8}{'train_err':<11}{'test_err':<11}{'#SV':<7}margin")
    for r in rows: print(f"{r[0]:<8g}{r[1]:<11.4f}{r[2]:<11.4f}{r[3]:<7}{r[4]:.2f}")
    plt.figure(figsize=(6, 4))
    plt.semilogx([r[0] for r in rows], [r[1] for r in rows], "o-", label="train error")
    plt.semilogx([r[0] for r in rows], [r[2] for r in rows], "s-", label="test error")
    plt.xlabel("C"); plt.ylabel("error"); plt.title(f"Dataset {k}: error vs C"); plt.legend(); plt.grid(True); plt.show()

**Observations to write up** (confirm against your table/plots):
- Small C → cheap margin violations → wide margin, many support vectors; if C is too small the model underfits (errors rise).
- Large C → narrow margin, few support vectors, behaves like the hard-margin SVM; beyond a point the solution stops changing.
- The closer the classes are (dataset 2 vs dataset 1), the larger C must be before the margin stops shrinking.

# Part 2 — Handwritten digit recognition (0 vs 1)

### Load and look at the images (equivalent of `strimage.m`)
Each line: `label idx:gray idx:gray …` with 1-based pixel indices; only non-zero pixels are listed. Label +1 = digit 1, −1 = digit 0.

In [ ]:
Xtr_img, ytr_img = load_svmlight_file(f_trimg, n_features=784, zero_based=False)
Xte_img, yte_img = load_svmlight_file(f_teimg, n_features=784, zero_based=False)
Xtr_img, Xte_img = Xtr_img.toarray() / 255.0, Xte_img.toarray() / 255.0       # dense, scaled to [0,1]
ytr_img, yte_img = np.where(ytr_img > 0, 1, -1), np.where(yte_img > 0, 1, -1)
print("train:", Xtr_img.shape, " ones:", (ytr_img == 1).sum(), " zeros:", (ytr_img == -1).sum())
print("test :", Xte_img.shape, " ones:", (yte_img == 1).sum(), " zeros:", (yte_img == -1).sum())

def show_digit(X, n, y=None, ax=None, title=None):
    """Display image number n (0-based) like strimage(n) in MATLAB."""
    ax = ax or plt.gca()
    ax.imshow(X[n].reshape(28, 28), cmap="gray_r"); ax.axis("off")
    ax.set_title(title if title else (f"#{n}" + (f" label={y[n]:+d}" if y is not None else "")), fontsize=9)

# sample images
fig, axs = plt.subplots(2, 8, figsize=(16, 4.5))
for j, i in enumerate(np.where(ytr_img == -1)[0][:8]): show_digit(Xtr_img, i, ytr_img, axs[0, j])
for j, i in enumerate(np.where(ytr_img == 1)[0][:8]):  show_digit(Xtr_img, i, ytr_img, axs[1, j])
plt.suptitle("Training samples (top: 0s, bottom: 1s)"); plt.show()

# average image of each class
fig, axs = plt.subplots(1, 2, figsize=(5, 2.8))
avg0, avg1 = Xtr_img[ytr_img == -1].mean(0), Xtr_img[ytr_img == 1].mean(0)
axs[0].imshow(avg0.reshape(28, 28), cmap="gray_r"); axs[0].set_title("average 0"); axs[0].axis("off")
axs[1].imshow(avg1.reshape(28, 28), cmap="gray_r"); axs[1].set_title("average 1"); axs[1].axis("off")
plt.show()

### 2(i) Linear SVM with no regularization (hard margin, ξᵢ = 0 ⇒ very large C)

In [ ]:
t = time.time()
hard = SVC(kernel="linear", C=1e6).fit(Xtr_img, ytr_img)
tr_err_hard, te_err_hard = err(hard, Xtr_img, ytr_img), err(hard, Xte_img, yte_img)
print(f"fit in {time.time()-t:.0f}s | #support vectors: {len(hard.support_)}")
print(f"Training error = {tr_err_hard:.5f}  ({int(round(tr_err_hard*len(ytr_img)))} of {len(ytr_img)})")
print(f"Test error     = {te_err_hard:.5f}  ({int(round(te_err_hard*len(yte_img)))} of {len(yte_img)})")

In [ ]:
# --- misclassified TEST images of the hard-margin SVM ---
tmis = np.where(hard.predict(Xte_img) != yte_img)[0]
print("Misclassified test images:", tmis)
if len(tmis):
    fig, axs = plt.subplots(1, len(tmis), figsize=(3*len(tmis), 3), squeeze=False)
    for ax, i in zip(axs[0], tmis):
        show_digit(Xte_img, i, ax=ax, title=f"test #{i}: true {yte_img[i]:+d}, pred {hard.predict(Xte_img[i:i+1])[0]:+d}")
    plt.show()

# --- the hard-margin SVM has no training mistakes when the data is separable, so to study a training
#     failure we take a strongly regularized SVM (tiny C) and pick one of ITS misclassified images ---
soft_small = SVC(kernel="linear", C=1e-4).fit(Xtr_img, ytr_img)
mis = np.where(soft_small.predict(Xtr_img) != ytr_img)[0]
print(f"\nHard margin: {int(round(tr_err_hard*len(ytr_img)))} training mistakes. "
      f"Soft margin with C=1e-4: {len(mis)} training mistakes.")
if len(mis):
    i = mis[0]
    f_val = soft_small.decision_function(Xtr_img[i:i+1])[0]
    fig, axs = plt.subplots(1, 3, figsize=(9, 3.2))
    show_digit(Xtr_img, i, ax=axs[0], title=f"train #{i}: true {ytr_img[i]:+d}, pred {soft_small.predict(Xtr_img[i:i+1])[0]:+d}")
    axs[1].imshow(avg0.reshape(28,28), cmap="gray_r"); axs[1].set_title("average 0"); axs[1].axis("off")
    axs[2].imshow(avg1.reshape(28,28), cmap="gray_r"); axs[2].set_title("average 1"); axs[2].axis("off")
    plt.show()
    print(f"Image #{i}: f(x) = {f_val:.2f} (C=1e-4); inked pixels = {(Xtr_img[i] > 0).sum()}  "
          f"(average 0: {(Xtr_img[ytr_img==-1] > 0).sum(1).mean():.0f}, average 1: {(Xtr_img[ytr_img==1] > 0).sum(1).mean():.0f})")
    print("Hard-margin SVM on this image: pred =", hard.predict(Xtr_img[i:i+1])[0], " y*f(x) =", ytr_img[i]*hard.decision_function(Xtr_img[i:i+1])[0])

In [ ]:
# --- why? look at the weight vector w and the training images closest to the margin ---
w = hard.coef_[0]
plt.figure(figsize=(4.5, 4)); v = np.abs(w).max()
plt.imshow(w.reshape(28, 28), cmap="bwr", vmin=-v, vmax=v); plt.colorbar()
plt.title("hard-margin w  (red: pushes to 1, blue: pushes to 0)"); plt.axis("off"); plt.show()

margins = ytr_img * hard.decision_function(Xtr_img)
print("smallest functional margin y*f(x) on training set:", margins.min())
hardest = np.argsort(margins)[:4]
fig, axs = plt.subplots(1, 4, figsize=(12, 3))
for ax, i in zip(axs, hardest): show_digit(Xtr_img, i, ax=ax, title=f"#{i} (label {ytr_img[i]:+d}) y·f={margins[i]:.2f}")
plt.show()

**How to explain it:** a linear SVM only computes a weighted sum of pixel intensities (the weight image above: positive in the centre where a ‘1’ has ink, negative in a ring where a ‘0’ has its strokes). Atypical images – very thin 0s, very thick or slanted 1s, broken digits – lie close to or across the boundary. If the hard-margin SVM has 0 training errors, say that the 0/1 set is linearly separable in 784-D and use the small-C model (above) to illustrate a failure.

### 2(ii) Experiment with different C

Range reasoning: for the hard-margin solution the largest αᵢ is below about 0.4 (check below). For any C larger than that the box constraint is never active, so the soft-margin solution equals the hard-margin one. So the interesting range is below that value, down to where errors appear (~1e-4).

In [ ]:
print("largest dual variable alpha of the hard-margin solution:", np.abs(hard.dual_coef_).max())
Cs2 = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1, 10]
tr_e, te_e, nsv = [], [], []
print(f"{'C':<8}{'train_err':<18}{'test_err':<18}#SV")
for c in Cs2:
    m = SVC(kernel="linear", C=c).fit(Xtr_img, ytr_img)
    a, b_ = err(m, Xtr_img, ytr_img), err(m, Xte_img, yte_img)
    tr_e.append(a); te_e.append(b_); nsv.append(len(m.support_))
    print(f"{c:<8g}{a:.5f} ({int(round(a*len(ytr_img)))}){'':<4}{b_:.5f} ({int(round(b_*len(yte_img)))}){'':<4}{nsv[-1]}")

plt.figure(figsize=(7.5, 4.5))
plt.semilogx(Cs2, np.array(tr_e)*100, "o-", label="training error")
plt.semilogx(Cs2, np.array(te_e)*100, "s-", label="test error")
plt.axhline(te_err_hard*100, color="gray", ls=":", label="hard-margin test error")
plt.xlabel("C"); plt.ylabel("error (%)"); plt.title("Digits 0/1: error vs C"); plt.legend(); plt.grid(True, which="both", alpha=.3); plt.show()

# A. best training error
bestA = Cs2[int(np.argmin(tr_e))]
print("A) C with the best (lowest) training error:", bestA, "-> training error", min(tr_e))
# B. its test error vs part (i)
print("B) test error at that C:", te_e[Cs2.index(bestA)], "| hard-margin test error from 2(i):", te_err_hard)
print("   lowest test error on the grid (hindsight only):", min(te_e), "at C =", [c for c,e in zip(Cs2,te_e) if e==min(te_e)])

### 2(ii)-C Can C be optimized for the test error?
Yes, but choosing C by looking at the test set makes the reported test error optimistic. The correct way is cross-validation on the **training set only**, then evaluating on the test set once.

In [ ]:
skf = StratifiedKFold(5, shuffle=True, random_state=0)
cv_err = []
for c in Cs2:
    s = cross_val_score(SVC(kernel="linear", C=c), Xtr_img, ytr_img, cv=skf, n_jobs=-1)
    cv_err.append(1 - s.mean()); print(f"C={c:<8g} 5-fold CV error = {cv_err[-1]:.5f}")
bestC = Cs2[int(np.argmin(cv_err))]
final = SVC(kernel="linear", C=bestC).fit(Xtr_img, ytr_img)
print(f"\nC chosen by cross-validation: {bestC}")
print(f"Test error with this C: {err(final, Xte_img, yte_img):.5f}  ({int(round(err(final, Xte_img, yte_img)*len(yte_img)))} of {len(yte_img)} images)")

# Part 3 — Kernel SVM (RBF),  K(xᵢ,xⱼ) = exp(−γ‖xᵢ−xⱼ‖²)

In [ ]:
X3, y3 = load_xy(f_tr3)
plt.figure(figsize=(6, 5))
plt.scatter(X3[y3==1,0],  X3[y3==1,1],  c="tab:blue", s=16, label="positive (+1)")
plt.scatter(X3[y3==-1,0], X3[y3==-1,1], c="tab:red", marker="x", s=16, label="negative (-1)")
plt.legend(); plt.title("Dataset 3: positives vs negatives"); plt.show()

lin = SVC(kernel="linear", C=1000).fit(X3, y3)
e = err(lin, X3, y3)
print(f"Linear SVM (C=1000) training error: {e:.4f} ({int(round(e*len(y3)))} of {len(y3)} points) -> no linear decision boundary separates the classes")
plt.figure(figsize=(6, 5)); plot_boundary(lin, X3, y3, "Linear SVM on dataset 3", margin=False); plt.show()

rbf = SVC(kernel="rbf", gamma=100, C=1).fit(X3, y3)
plt.figure(figsize=(6, 5)); plot_boundary(rbf, X3, y3, f"RBF SVM, gamma=100, C=1 (train err={err(rbf, X3, y3):.4f})", margin=False); plt.show()

fig, axs = plt.subplots(1, 4, figsize=(20, 4.8))
print(f"{'gamma':<8}{'train_err':<11}{'#misclassified':<16}#SV")
for g, ax in zip([1, 10, 100, 1000], axs):
    m = SVC(kernel="rbf", gamma=g, C=1).fit(X3, y3)
    e = err(m, X3, y3)
    plot_boundary(m, X3, y3, f"gamma={g} (error {e:.3f})", ax=ax, margin=False)
    print(f"{g:<8}{e:<11.4f}{int(round(e*len(y3))):<16}{len(m.support_)}")
plt.tight_layout(); plt.show()

**Observation on γ:** small γ (1) → very wide kernels, almost linear boundary, underfits. γ = 10–100 → boundary bends with the class border. γ = 1000 → each point only influences its immediate neighbourhood, the boundary breaks into small islands, training error → 0 with almost all points as support vectors — overfitting. Training error alone always favours the largest γ, so γ (and C) should be chosen by cross-validation.